# ⚽ Football Analysis — Inference toàn diện (Ảnh & Video)

Notebook này gộp toàn bộ pipeline của bạn thành **một luồng inference duy nhất** cho cả **ảnh** lẫn **video**, hiển thị đồng thời:

- **Khung hình gốc** được annotate: **ellipse** dưới chân cầu thủ / thủ môn / trọng tài, **tam giác** trên quả bóng.
- **2D mapping (radar)** kiểu FIFA: đặt **ở giữa, sát mép dưới** khung hình.
- **Biểu đồ Voronoi**: đặt **bên cạnh** (góc dưới phải).

Lần này vẫn **load tham số mô hình trực tiếp từ Roboflow** (như bạn đang làm). Lần sau muốn chạy bằng weights tải về Drive, chỉ cần thay 2 hàm load model ở mục *Load models*.

> **Thứ tự chạy:** chạy lần lượt từ trên xuống. Phần *Helpers* định nghĩa hàm, phần *Inference* mới thực sự chạy.

## 1. Cài đặt & cấu hình môi trường

In [ ]:
# (Tuỳ chọn) mount Drive nếu bạn muốn đọc/ghi file từ Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
import os
os.environ["ONNXRUNTIME_EXECUTION_PROVIDERS"] = "[CUDAExecutionProvider]"

from google.colab import userdata
# Lấy API key từ Colab Secrets (biểu tượng chìa khoá bên trái)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

In [ ]:
# Cài dependencies (chạy 1 lần; có thể cần Restart runtime sau bước này)
!pip install -q inference-gpu
!pip install -q git+https://github.com/roboflow/sports.git
!pip install -q supervision umap-learn more-itertools gdown

In [ ]:
# (Tuỳ chọn) tải vài video mẫu để test nhanh
!gdown -O "2e57b9_0.mp4" "https://drive.google.com/uc?id=19PGw55V8aA6GZu5-Aac5_9mCy3fNxmEf"


## 2. Import & cấu hình chung

In [ ]:
import cv2
import numpy as np
import supervision as sv
import torch
from tqdm import tqdm
from typing import Optional

from inference import get_model
from sports.common.view import ViewTransformer
from sports.common.team import TeamClassifier
from sports.configs.soccer import SoccerPitchConfiguration
from sports.annotators.soccer import (
    draw_pitch,
    draw_points_on_pitch,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

# ----- Class id của model detection (ball, player, referee, goalkeeper) -----
BALL_ID = 0
GOALKEEPER_ID = 1
PLAYER_ID = 2
REFEREE_ID = 3

# ----- Ngưỡng -----
CONF = 0.3          # confidence cho cả 2 model
KP_CONF = 0.5       # ngưỡng confidence cho pitch keypoint khi tính homography
NMS_THRESHOLD = 0.5

# ----- Màu (BGR-friendly hex) -----
COLOR_TEAM_1 = '00BFFF'   # xanh
COLOR_TEAM_2 = 'FF1493'   # hồng
COLOR_REFEREE = 'FFD700'  # vàng

CONFIG = SoccerPitchConfiguration()

## 3. Load models (từ Roboflow)

> Lần sau muốn dùng weights local trên Drive, thay 2 dòng dưới bằng:
> ```python
> from ultralytics import YOLO
> PLAYER_DETECTION_MODEL = YOLO('/content/drive/MyDrive/.../players.pt')
> FIELD_DETECTION_MODEL  = YOLO('/content/drive/MyDrive/.../pitch.pt')
> ```
> Phần code phía sau dùng `.infer(frame, confidence=...)` nên tương thích cả 2 cách.

In [ ]:
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')

PLAYER_DETECTION_MODEL_ID = "soccernet-1ae2v-e6wqy/3"
FIELD_DETECTION_MODEL_ID  = "football-field-detection-f07vi/15"

PLAYER_DETECTION_MODEL = get_model(model_id=PLAYER_DETECTION_MODEL_ID, api_key=ROBOFLOW_API_KEY)
FIELD_DETECTION_MODEL  = get_model(model_id=FIELD_DETECTION_MODEL_ID,  api_key=ROBOFLOW_API_KEY)
print('✅ Đã load 2 model.')

## 4. Annotators

In [ ]:
ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex([f'#{COLOR_TEAM_1}', f'#{COLOR_TEAM_2}', f'#{COLOR_REFEREE}']),
    thickness=2,
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex([f'#{COLOR_TEAM_1}', f'#{COLOR_TEAM_2}', f'#{COLOR_REFEREE}']),
    text_color=sv.Color.from_hex('#000000'),
    text_position=sv.Position.BOTTOM_CENTER,
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex(f'#{COLOR_REFEREE}'),
    base=25, height=21, outline_thickness=1,
)

## 5. Helpers — Voronoi mượt & gán đội cho thủ môn

In [ ]:
def resolve_goalkeepers_team_id(players: sv.Detections,
                                goalkeepers: sv.Detections) -> np.ndarray:
    """Gán thủ môn về đội gần centroid hơn."""
    gk_xy = goalkeepers.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    pl_xy = players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    team0 = pl_xy[players.class_id == 0].mean(axis=0)
    team1 = pl_xy[players.class_id == 1].mean(axis=0)
    out = []
    for xy in gk_xy:
        out.append(0 if np.linalg.norm(xy - team0) < np.linalg.norm(xy - team1) else 1)
    return np.array(out)

In [ ]:
def draw_pitch_voronoi_diagram_2(
    config: SoccerPitchConfiguration,
    team_1_xy: np.ndarray,
    team_2_xy: np.ndarray,
    team_1_color: sv.Color = sv.Color.RED,
    team_2_color: sv.Color = sv.Color.WHITE,
    opacity: float = 0.5,
    padding: int = 50,
    scale: float = 0.1,
    pitch: Optional[np.ndarray] = None,
) -> np.ndarray:
    """Voronoi với chuyển màu mượt giữa 2 vùng kiểm soát."""
    if pitch is None:
        pitch = draw_pitch(config=config, padding=padding, scale=scale)

    scaled_width = int(config.width * scale)
    scaled_length = int(config.length * scale)
    voronoi = np.zeros_like(pitch, dtype=np.uint8)

    c1 = np.array(team_1_color.as_bgr(), dtype=np.uint8)
    c2 = np.array(team_2_color.as_bgr(), dtype=np.uint8)

    ys, xs = np.indices((scaled_width + 2 * padding, scaled_length + 2 * padding))
    ys -= padding
    xs -= padding

    def dist(xy):
        return np.sqrt((xy[:, 0][:, None, None] * scale - xs) ** 2 +
                       (xy[:, 1][:, None, None] * scale - ys) ** 2)

    d1 = np.min(dist(team_1_xy), axis=0)
    d2 = np.min(dist(team_2_xy), axis=0)

    steepness = 15
    ratio = d2 / np.clip(d1 + d2, a_min=1e-5, a_max=None)
    blend = np.tanh((ratio - 0.5) * steepness) * 0.5 + 0.5

    for c in range(3):
        voronoi[:, :, c] = (blend * c1[c] + (1 - blend) * c2[c]).astype(np.uint8)

    return cv2.addWeighted(voronoi, opacity, pitch, 1 - opacity, 0)

## 6. Helpers — Dựng radar 2D, panel Voronoi & ghép overlay

`compose_frame` là phần quyết định **bố cục kiểu FIFA**: radar ở giữa sát đáy, Voronoi bên cạnh (góc dưới phải). Bạn có thể chỉnh các tham số `*_scale`, `margin`, `alpha`, vị trí Voronoi (`voronoi_side`) tuỳ thích.

In [ ]:
def build_radar(ball_xy, team0_xy, team1_xy, ref_xy):
    """Radar 2D kiểu minimap."""
    pitch = draw_pitch(CONFIG)
    if len(team0_xy):
        pitch = draw_points_on_pitch(CONFIG, xy=team0_xy,
            face_color=sv.Color.from_hex(COLOR_TEAM_1), edge_color=sv.Color.BLACK,
            radius=16, pitch=pitch)
    if len(team1_xy):
        pitch = draw_points_on_pitch(CONFIG, xy=team1_xy,
            face_color=sv.Color.from_hex(COLOR_TEAM_2), edge_color=sv.Color.BLACK,
            radius=16, pitch=pitch)
    if len(ref_xy):
        pitch = draw_points_on_pitch(CONFIG, xy=ref_xy,
            face_color=sv.Color.from_hex(COLOR_REFEREE), edge_color=sv.Color.BLACK,
            radius=16, pitch=pitch)
    if len(ball_xy):
        pitch = draw_points_on_pitch(CONFIG, xy=ball_xy,
            face_color=sv.Color.WHITE, edge_color=sv.Color.BLACK,
            radius=10, pitch=pitch)
    return pitch


def build_voronoi(team0_xy, team1_xy, ball_xy):
    """Panel Voronoi (nền trắng cho dễ nhìn)."""
    pitch = draw_pitch(CONFIG, background_color=sv.Color.WHITE, line_color=sv.Color.BLACK)
    pitch = draw_pitch_voronoi_diagram_2(
        CONFIG, team_1_xy=team0_xy, team_2_xy=team1_xy,
        team_1_color=sv.Color.from_hex(COLOR_TEAM_1),
        team_2_color=sv.Color.from_hex(COLOR_TEAM_2),
        pitch=pitch)
    if len(team0_xy):
        pitch = draw_points_on_pitch(CONFIG, xy=team0_xy,
            face_color=sv.Color.from_hex(COLOR_TEAM_1), edge_color=sv.Color.WHITE,
            radius=16, thickness=1, pitch=pitch)
    if len(team1_xy):
        pitch = draw_points_on_pitch(CONFIG, xy=team1_xy,
            face_color=sv.Color.from_hex(COLOR_TEAM_2), edge_color=sv.Color.WHITE,
            radius=16, thickness=1, pitch=pitch)
    if len(ball_xy):
        pitch = draw_points_on_pitch(CONFIG, xy=ball_xy,
            face_color=sv.Color.WHITE, edge_color=sv.Color.WHITE,
            radius=8, thickness=1, pitch=pitch)
    return pitch

In [ ]:
def _resize_w(img, width):
    h, w = img.shape[:2]
    return cv2.resize(img, (width, int(h * width / w)), interpolation=cv2.INTER_AREA)


def _overlay(scene, panel, x, y, alpha):
    """Dán panel lên scene tại (x,y) với độ mờ alpha, tự clamp trong biên."""
    H, W = scene.shape[:2]
    ph, pw = panel.shape[:2]
    if pw >= W or ph >= H:
        return scene
    x = max(0, min(x, W - pw))
    y = max(0, min(y, H - ph))
    roi = scene[y:y + ph, x:x + pw]
    scene[y:y + ph, x:x + pw] = cv2.addWeighted(panel, alpha, roi, 1 - alpha, 0)
    return scene


def compose_frame(annotated, radar=None, voronoi=None,
                  radar_scale=0.40, voronoi_scale=0.26,
                  alpha=0.85, margin=20, voronoi_side='right'):
    """Bố cục FIFA: radar giữa-đáy, voronoi bên cạnh."""
    out = annotated.copy()
    H, W = out.shape[:2]

    if voronoi is not None:
        v = _resize_w(voronoi, int(W * voronoi_scale))
        vh, vw = v.shape[:2]
        vx = (W - vw - margin) if voronoi_side == 'right' else margin
        out = _overlay(out, v, vx, H - vh - margin, alpha)

    if radar is not None:
        r = _resize_w(radar, int(W * radar_scale))
        rh, rw = r.shape[:2]
        out = _overlay(out, r, (W - rw) // 2, H - rh - margin, alpha)

    return out

## 6.5. Helpers — chống flicker & quỹ đạo bóng

- **`BallTracker`** (image space): chọn detection gần vị trí trước nhất (loại box nhiễu ở xa), **coast** giữ vị trí khi mất bóng vài frame, và làm mượt EMA → hết nhấp nháy tam giác.
- **`Ball2DPathSmoother`** (pitch space, đơn vị cm): **giới hạn vận tốc** mỗi frame + EMA. Khi bóng bay bổng, điểm chiếu trên map vọt ra xa; clamp vận tốc khiến chấm bóng đi thẳng về điểm rơi thay vì vẽ vòng cung.
- **`SpeedEstimator`** (pitch space): tốc độ tức thời (km/h) theo `tracker_id`, đo trên cửa sổ vài frame để bớt nhiễu rồi hiển thị dưới chân cầu thủ.

In [ ]:
class BallTracker:
    """Chống flicker + lấp khoảng trống cho bóng (toạ độ ảnh)."""
    def __init__(self, max_coast=8, max_jump=250, ema=0.5):
        self.max_coast = max_coast   # số frame tối đa được giữ khi mất bóng
        self.max_jump = max_jump     # px: loại detection nhảy quá xa so với vị trí trước
        self.ema = ema
        self.xy = None               # tâm bóng đã mượt
        self.wh = None               # kích thước box gần nhất
        self.miss = 0

    def update(self, ball_detections):
        """Trả về xyxy dạng (1,4) để vẽ/transform, hoặc None nếu mất bóng quá lâu."""
        cand = None
        if len(ball_detections):
            xyxy = ball_detections.xyxy
            cxcy = (xyxy[:, :2] + xyxy[:, 2:]) / 2
            if self.xy is None:
                idx = int(np.argmax(ball_detections.confidence))  # lần đầu: conf cao nhất
            else:
                d = np.linalg.norm(cxcy - self.xy, axis=1)
                idx = int(np.argmin(d))
                if d[idx] > self.max_jump:   # nhảy quá xa -> coi như nhiễu
                    idx = None
            if idx is not None:
                cand = xyxy[idx]

        if cand is not None:
            c = (cand[:2] + cand[2:]) / 2
            wh = cand[2:] - cand[:2]
            if self.xy is None:
                self.xy, self.wh = c, wh
            else:
                self.xy = self.ema * c + (1 - self.ema) * self.xy
                self.wh = self.ema * wh + (1 - self.ema) * self.wh
            self.miss = 0
        else:
            self.miss += 1
            if self.xy is None or self.miss > self.max_coast:
                return None              # mất bóng -> không vẽ

        x, y = self.xy; w, h = self.wh
        return np.array([[x - w / 2, y - h / 2, x + w / 2, y + h / 2]])


class Ball2DPathSmoother:
    """Làm mượt + giới hạn vận tốc cho bóng trên map 2D (pitch coords, cm).

    max_step: bước tối đa mỗi frame (cm). Sân dài 12000cm @ ~25fps.
    Bóng sệt nhanh ~120-150 cm/frame; vòng cung do bóng bổng thường lớn hơn -> bị clamp.
    """
    def __init__(self, max_step=150.0, ema=0.4, max_coast=8):
        self.max_step = max_step
        self.ema = ema
        self.max_coast = max_coast
        self.last = None
        self.miss = 0

    def update(self, pitch_ball_xy):
        """pitch_ball_xy: (N,2) hoặc rỗng. Trả về (1,2) hoặc None."""
        meas = pitch_ball_xy[0] if len(pitch_ball_xy) else None
        if meas is None:
            self.miss += 1
            if self.last is None or self.miss > self.max_coast:
                return None
            return self.last[None, :]
        self.miss = 0
        if self.last is None:
            self.last = meas
        else:
            delta = meas - self.last
            dist = np.linalg.norm(delta)
            if dist > self.max_step:               # clamp vận tốc -> chặn vòng cung
                delta = delta / dist * self.max_step
            target = self.last + delta
            self.last = self.ema * target + (1 - self.ema) * self.last
        return self.last[None, :]

In [ ]:
class SpeedEstimator:
    """Tốc độ tức thời (km/h) theo tracker_id, đo trên toạ độ sân (cm).

    Tính quãng đường giữa frame cũ nhất và mới nhất trong cửa sổ -> ổn định hơn
    so với chênh lệch 2 frame liền kề (vốn rất nhiễu do detection/homography rung).
    """
    def __init__(self, fps, window=5, ema=0.6, max_kmh=40.0, max_gap=15):
        self.fps = fps
        self.window = window      # số frame trong cửa sổ đo
        self.ema = ema
        self.max_kmh = max_kmh    # chặn giá trị phi lý do rung
        self.max_gap = max_gap    # mất dấu quá lâu -> reset, không tính
        self.hist = defaultdict(lambda: deque(maxlen=window))
        self.speed = {}

    def update(self, tracker_ids, pitch_xy, frame_idx):
        out = {}
        for tid, xy in zip(tracker_ids, pitch_xy):
            tid = int(tid)
            h = self.hist[tid]
            if h and frame_idx - h[-1][0] > self.max_gap:
                h.clear()                       # xuất hiện lại sau khi mất dấu lâu
            h.append((frame_idx, np.asarray(xy, dtype=float)))
            if len(h) >= 2:
                f0, xy0 = h[0]
                f1, xy1 = h[-1]
                df = f1 - f0
                if df > 0:
                    dist_m = float(np.linalg.norm(xy1 - xy0)) / 100.0   # cm -> m
                    v = dist_m / (df / self.fps) * 3.6                  # m/s -> km/h
                    v = min(v, self.max_kmh)
                    v = self.ema * v + (1 - self.ema) * self.speed.get(tid, v)
                    self.speed[tid] = v
            out[tid] = self.speed.get(tid, 0.0)
        return out

## 7. Hàm xử lý 1 khung hình

Trả về ảnh composite (annotate + radar + voronoi). Dùng chung cho cả ảnh và video.

In [ ]:
def process_frame(frame, team_resolver, tracker,
                  ball_tracker=None, ball_2d=None,
                  speed_estimator=None, frame_idx=0, draw_overlays=True):
    H, W = frame.shape[:2]

    # 1) Detection cầu thủ / bóng / trọng tài
    result = PLAYER_DETECTION_MODEL.infer(frame, confidence=CONF)[0]
    detections = sv.Detections.from_inference(result)

    # 2) Bóng: ổn định qua các frame (chống flicker / lấp khi mất bóng)
    ball_detections = detections[detections.class_id == BALL_ID]
    if ball_tracker is not None:
        ball_xyxy = ball_tracker.update(ball_detections)
    else:
        ball_xyxy = ball_detections.xyxy if len(ball_detections) else None
    if ball_xyxy is not None:
        ball_for_draw = sv.Detections(
            xyxy=sv.pad_boxes(xyxy=ball_xyxy, px=10),
            class_id=np.array([BALL_ID]),
        )
    else:
        ball_for_draw = sv.Detections.empty()

    # 3) Người: NMS + tracking
    others = detections[detections.class_id != BALL_ID]
    others = others.with_nms(threshold=NMS_THRESHOLD, class_agnostic=True)
    others = tracker.update_with_detections(detections=others)

    goalkeepers = others[others.class_id == GOALKEEPER_ID]
    players = others[others.class_id == PLAYER_ID]
    referees = others[others.class_id == REFEREE_ID]

    # 4) Gán đội (làm mượt theo tracker_id để chống flicker)
    if len(players):
        players.class_id = team_resolver.predict_players(frame, players)
    if len(goalkeepers) and len(players):
        resolved = resolve_goalkeepers_team_id(players, goalkeepers)
        goalkeepers.class_id = team_resolver.smooth_goalkeepers(goalkeepers, resolved)
    if len(referees):
        referees.class_id = referees.class_id - 1  # 3 -> 2 (màu vàng)

    # 5) Homography 1 lần (dùng chung cho tốc độ + radar + voronoi)
    transformer = None
    if draw_overlays or speed_estimator is not None:
        try:
            fkp = FIELD_DETECTION_MODEL.infer(frame, confidence=CONF)[0]
            key_points = sv.KeyPoints.from_inference(fkp)
            mask = key_points.confidence[0] > KP_CONF
            if mask.sum() >= 4:
                transformer = ViewTransformer(
                    source=key_points.xy[0][mask],
                    target=np.array(CONFIG.vertices)[mask])
        except Exception as e:
            if not getattr(process_frame, "_warned", False):
                print("⚠️ Bỏ qua overlay/tốc độ ở 1 frame:", repr(e))
                process_frame._warned = True
            transformer = None

    def to_pitch(det):
        if transformer is None or not len(det):
            return np.empty((0, 2))
        xy = det.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
        return transformer.transform_points(points=xy)

    # players + goalkeepers (giữ tracker_id) -> toạ độ sân
    pg = sv.Detections.merge([d for d in (players, goalkeepers) if len(d)]) \
         if (len(players) or len(goalkeepers)) else sv.Detections.empty()
    pitch_pg = to_pitch(pg)

    # 6) Tốc độ tức thời (km/h) theo tracker_id
    speeds = {}
    if speed_estimator is not None and len(pitch_pg) and pg.tracker_id is not None:
        speeds = speed_estimator.update(pg.tracker_id, pitch_pg, frame_idx)

    # 7) Annotate khung gốc (ellipse + label tốc độ + tam giác bóng)
    parts = [d for d in (players, goalkeepers, referees) if len(d)]
    annotated = frame.copy()
    if parts:
        merged = sv.Detections.merge(parts)
        merged.class_id = merged.class_id.astype(int)
        annotated = ellipse_annotator.annotate(scene=annotated, detections=merged)
        if merged.tracker_id is not None and len(merged.tracker_id):
            labels = []
            for tid in merged.tracker_id:
                v = speeds.get(int(tid))
                labels.append(f"{v:.0f} km/h" if v is not None else f"#{int(tid)}")
            annotated = label_annotator.annotate(scene=annotated, detections=merged, labels=labels)
    if len(ball_for_draw):
        annotated = triangle_annotator.annotate(scene=annotated, detections=ball_for_draw)

    if not draw_overlays:
        return annotated

    # 8) Radar + Voronoi (tái dùng pitch_pg đã tính)
    radar = voronoi = None
    if transformer is not None:
        pitch_ball = to_pitch(ball_for_draw)
        if ball_2d is not None:
            pb = ball_2d.update(pitch_ball)
            pitch_ball = pb if pb is not None else np.empty((0, 2))
        pitch_ref = to_pitch(referees)
        if len(pitch_pg):
            cid = pg.class_id.astype(int)
            team0 = pitch_pg[cid == 0]
            team1 = pitch_pg[cid == 1]
        else:
            team0 = team1 = np.empty((0, 2))
        radar = build_radar(pitch_ball, team0, team1, pitch_ref)
        voronoi = build_voronoi(team0, team1, pitch_ball)

    return compose_frame(annotated, radar=radar, voronoi=voronoi)

## 8. TeamClassifier + chống flicker

Phân đội bị nhấp nháy vì `predict` chạy **độc lập từng frame**. Hai cải tiến ở đây:

1. **`torso_crop`** — chỉ lấy vùng thân áo (nửa trên bounding box) thay vì cả người, vì màu áo mới là đặc trưng phân đội; chân/cỏ/quần là nhiễu. Dùng **cùng một cách cắt** ở cả lúc *fit* lẫn lúc *predict*.
2. **`TeamResolver`** — bỏ phiếu đa số theo `tracker_id` trên một cửa sổ thời gian, nên một frame nhận nhầm không đủ sức lật nhãn của một track đã ổn định. Áp cho **cả cầu thủ lẫn thủ môn**.

Lưu ý fit: với video gom crop qua nhiều frame nên cụm 2 đội ổn định; với ảnh đơn lẻ nếu ít cầu thủ thì nên fit trên một video cùng trận rồi tái dùng.

In [ ]:
def torso_crop(frame, xyxy, top=0.0, bottom=0.55):
    """Cắt vùng thân áo (nửa trên box) — đặc trưng tốt hơn để phân đội."""
    x1, y1, x2, y2 = xyxy
    h = y2 - y1
    return sv.crop_image(frame, [x1, y1 + h * top, x2, y1 + h * bottom])

In [ ]:
def fit_team_classifier(source_path, is_video, stride=20):
    crops = []
    if is_video:
        gen = sv.get_video_frames_generator(source_path, stride=stride)
        frames = tqdm(gen, desc='collecting crops')
    else:
        frames = [cv2.imread(source_path)]

    for frame in frames:
        res = PLAYER_DETECTION_MODEL.infer(frame, confidence=CONF)[0]
        det = sv.Detections.from_inference(res)
        det = det[det.class_id == PLAYER_ID]          # chỉ lấy cầu thủ
        crops += [torso_crop(frame, xyxy) for xyxy in det.xyxy]   # cắt thân áo

    print(f'Số crop thu được: {len(crops)}')
    tc = TeamClassifier(device=DEVICE)
    tc.fit(crops)
    return tc

In [ ]:
from collections import defaultdict, deque, Counter

class TeamResolver:
    """Bọc TeamClassifier, làm mượt nhãn đội theo tracker_id để chống flicker.

    - window: số frame gần nhất dùng để bỏ phiếu (vd 1 giây = fps frame).
    - switch_ratio: hysteresis — chỉ đổi nhãn khi đội mới chiếm > tỉ lệ này
      trong cửa sổ (0.5 = đa số đơn thuần; tăng lên 0.6 để "lì" hơn).
    """
    def __init__(self, classifier, window=30, crop_fn=torso_crop, switch_ratio=0.5):
        self.clf = classifier
        self.window = window
        self.crop_fn = crop_fn
        self.switch_ratio = switch_ratio
        self.history = defaultdict(lambda: deque(maxlen=window))
        self.last = {}   # nhãn ổn định gần nhất của mỗi track

    def _vote(self, tids, raw):
        out = []
        for tid, p in zip(tids, raw):
            tid = int(tid)
            h = self.history[tid]
            h.append(int(p))
            label, count = Counter(h).most_common(1)[0]
            prev = self.last.get(tid, label)
            # chỉ đổi nhãn nếu phe mới đủ áp đảo (hysteresis)
            if label != prev and count / len(h) < self.switch_ratio:
                label = prev
            self.last[tid] = label
            out.append(label)
        return np.array(out, dtype=int)

    def predict_players(self, frame, players):
        if not len(players):
            return np.array([], dtype=int)
        raw = self.clf.predict([self.crop_fn(frame, b) for b in players.xyxy])
        return self._vote(players.tracker_id, raw)

    def smooth_goalkeepers(self, goalkeepers, resolved):
        """resolved = kết quả resolve_goalkeepers_team_id của frame hiện tại."""
        if not len(goalkeepers):
            return np.array([], dtype=int)
        return self._vote(goalkeepers.tracker_id, resolved)

## 9. Inference trên ẢNH

Đổi `IMAGE_PATH` thành ảnh của bạn (upload lên Colab hoặc lấy 1 frame từ video).

In [ ]:
# Ví dụ: lấy 1 frame từ video làm ảnh test (hoặc thay bằng đường dẫn ảnh của bạn)
IMAGE_PATH = "/content/sample_frame.jpg"

_gen = sv.get_video_frames_generator("/content/2e57b9_0.mp4", start=200)
cv2.imwrite(IMAGE_PATH, next(_gen))
print('Đã tạo ảnh test:', IMAGE_PATH)

In [ ]:
OUTPUT_IMAGE_PATH = "/content/result_image.jpg"

frame = cv2.imread(IMAGE_PATH)

# fit team classifier ngay trên ảnh (xem lưu ý ở mục 8)
tc_img = fit_team_classifier(IMAGE_PATH, is_video=False)
resolver_img = TeamResolver(tc_img, window=1)   # ảnh: 1 frame nên không cần làm mượt

tracker = sv.ByteTrack(); tracker.reset()
composite = process_frame(frame, resolver_img, tracker,
                          ball_tracker=BallTracker(), ball_2d=Ball2DPathSmoother())

cv2.imwrite(OUTPUT_IMAGE_PATH, composite)
sv.plot_image(cv2.cvtColor(composite, cv2.COLOR_BGR2RGB))
print('Đã lưu:', OUTPUT_IMAGE_PATH)

(Tuỳ chọn) Xem riêng radar và Voronoi của ảnh để kiểm tra.

In [ ]:
# Dựng lại radar & voronoi đầy đủ cho ảnh để xem tách biệt
res = PLAYER_DETECTION_MODEL.infer(frame, confidence=CONF)[0]
det = sv.Detections.from_inference(res)
ball = det[det.class_id == BALL_ID]
others = det[det.class_id != BALL_ID].with_nms(threshold=NMS_THRESHOLD, class_agnostic=True)
gk = others[others.class_id == GOALKEEPER_ID]
pl = others[others.class_id == PLAYER_ID]
rf = others[others.class_id == REFEREE_ID]
if len(pl):
    pl.class_id = tc_img.predict([torso_crop(frame, b) for b in pl.xyxy])
if len(gk) and len(pl):
    gk.class_id = resolve_goalkeepers_team_id(pl, gk)
if len(rf):
    rf.class_id = rf.class_id - 1

fkp = FIELD_DETECTION_MODEL.infer(frame, confidence=CONF)[0]
kp = sv.KeyPoints.from_inference(fkp)
m = kp.confidence[0] > KP_CONF
tr = ViewTransformer(source=kp.xy[0][m], target=np.array(CONFIG.vertices)[m])
pg = sv.Detections.merge([d for d in (pl, gk) if len(d)])
pgxy = tr.transform_points(pg.get_anchors_coordinates(sv.Position.BOTTOM_CENTER))
ballxy = tr.transform_points(ball.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)) if len(ball) else np.empty((0,2))
refxy = tr.transform_points(rf.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)) if len(rf) else np.empty((0,2))
t0, t1 = pgxy[pg.class_id.astype(int)==0], pgxy[pg.class_id.astype(int)==1]

sv.plot_image(cv2.cvtColor(build_radar(ballxy, t0, t1, refxy), cv2.COLOR_BGR2RGB))
sv.plot_image(cv2.cvtColor(build_voronoi(t0, t1, ballxy), cv2.COLOR_BGR2RGB))

## 10. Inference trên VIDEO — pipeline 2 lượt (offline)

Vẽ vòng cung của bóng bay và tốc độ nhảy đều cần **thông tin tương lai** để xử lý đúng, nên video chạy **2 lượt** (inference vẫn chỉ 1 lần):

- **Lượt 1 — phân tích:** chạy model, **làm mượt homography** (EMA trên keypoint sân → toạ độ bớt rung), lưu dữ liệu gọn từng frame.
- **Lượt 2 — hậu kỳ + vẽ:**
  - *Bóng:* phát hiện đoạn bay/nhiễu (vận tốc trên sân vượt ngưỡng) rồi **nội suy thẳng** giữa điểm sút và điểm rơi → hết vòng cung.
  - *Tốc độ:* làm sạch cú nhảy id-swap + làm mượt vị trí theo từng track → km/h ổn định.

> Lượt 1 nặng (2 model + SigLIP/frame). Dùng `max_frames` để test trước.

In [ ]:
class HomographySmoother:
    """EMA trên keypoint sân rồi dựng lại homography mỗi frame -> giảm rung."""
    def __init__(self, alpha=0.3):
        self.alpha = alpha
        self.kp = None
        self.seen = None

    def update(self, frame):
        fkp = FIELD_DETECTION_MODEL.infer(frame, confidence=CONF)[0]
        kp = sv.KeyPoints.from_inference(fkp)
        xy = kp.xy[0]
        cur = kp.confidence[0] > KP_CONF
        if self.kp is None:
            self.kp = xy.astype(float).copy()
            self.seen = cur.copy()
        else:
            for i in range(len(xy)):
                if cur[i]:
                    if self.seen[i]:
                        self.kp[i] = self.alpha * xy[i] + (1 - self.alpha) * self.kp[i]
                    else:
                        self.kp[i] = xy[i]; self.seen[i] = True
        if cur.sum() < 4:
            return None
        return ViewTransformer(source=self.kp[cur], target=np.array(CONFIG.vertices)[cur])

In [ ]:
def analyze_video(source_path, team_resolver, max_frames=None):
    """Lượt 1: chạy model 1 lần, trả về (info, records) gọn nhẹ."""
    info = sv.VideoInfo.from_video_path(source_path)
    tracker = sv.ByteTrack(); tracker.reset()
    ball_tracker = BallTracker()
    homo = HomographySmoother()
    records = []
    total = info.total_frames if max_frames is None else min(max_frames, info.total_frames)
    gen = sv.get_video_frames_generator(source_path)
    for i, frame in enumerate(tqdm(gen, total=total, desc='analyzing')):
        if max_frames is not None and i >= max_frames:
            break
        det = sv.Detections.from_inference(PLAYER_DETECTION_MODEL.infer(frame, confidence=CONF)[0])

        ball = det[det.class_id == BALL_ID]
        ball_xyxy = ball_tracker.update(ball)

        others = det[det.class_id != BALL_ID].with_nms(threshold=NMS_THRESHOLD, class_agnostic=True)
        others = tracker.update_with_detections(detections=others)
        gk = others[others.class_id == GOALKEEPER_ID]
        pl = others[others.class_id == PLAYER_ID]
        rf = others[others.class_id == REFEREE_ID]
        if len(pl):
            pl.class_id = team_resolver.predict_players(frame, pl)
        if len(gk) and len(pl):
            gk.class_id = team_resolver.smooth_goalkeepers(gk, resolve_goalkeepers_team_id(pl, gk))
        if len(rf):
            rf.class_id = rf.class_id - 1

        transformer = homo.update(frame)

        parts = [d for d in (pl, gk, rf) if len(d)]
        if parts:
            merged = sv.Detections.merge(parts)
            xyxy, cid, tid = merged.xyxy, merged.class_id.astype(int), merged.tracker_id
            isref = np.concatenate([np.zeros(len(pl), bool), np.zeros(len(gk), bool),
                                    np.ones(len(rf), bool)])
            if transformer is not None:
                foot = merged.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
                pitch = transformer.transform_points(points=foot)
            else:
                pitch = np.full((len(merged), 2), np.nan)
        else:
            xyxy = np.empty((0, 4)); cid = np.empty((0,), int); tid = np.empty((0,), int)
            isref = np.empty((0,), bool); pitch = np.empty((0, 2))

        ball_pitch = None
        if transformer is not None and ball_xyxy is not None:
            bc = np.array([[(ball_xyxy[0, 0] + ball_xyxy[0, 2]) / 2, ball_xyxy[0, 3]]])
            ball_pitch = transformer.transform_points(points=bc)[0]

        records.append(dict(xyxy=xyxy, cid=cid, tid=tid, isref=isref, pitch=pitch,
                            ball_box=(None if ball_xyxy is None else ball_xyxy[0]),
                            ball_pitch=ball_pitch))
    return info, records

In [ ]:
def smooth_ball_path(records, fps, touch_px=70.0):
    """Lượt 2a: bóng trên map = đường gấp khúc nối các lần CHẠM BÓNG.

    Tại lần chạm, bóng ở sát mặt sân (gần bàn chân cầu thủ) -> homography đáng tin.
    Giữa hai lần chạm (kể cả khi bóng bay) -> nội suy thẳng => xoá vòng cung.
    Phát hiện chạm bằng khoảng cách bóng -> BÀN CHÂN gần nhất trong ảnh: bóng đang
    bay nằm cao trong khung hình nên xa bàn chân -> không bị nhầm là chạm.

    touch_px: ngưỡng khoảng cách (px) ~ cho video 1080p; chỉnh theo độ phân giải.
    """
    F = len(records)
    raw = [r['ball_pitch'] for r in records]
    detected = [f for f, p in enumerate(raw) if p is not None]
    if len(detected) < 2:
        return raw

    touch = []
    for f, r in enumerate(records):
        bb = r['ball_box']
        if raw[f] is None or bb is None or len(r['xyxy']) == 0:
            continue
        bcx = (bb[0] + bb[2]) / 2.0
        bcy = (bb[1] + bb[3]) / 2.0
        foot_x = (r['xyxy'][:, 0] + r['xyxy'][:, 2]) / 2.0
        foot_y = r['xyxy'][:, 3]
        if np.min(np.hypot(foot_x - bcx, foot_y - bcy)) < touch_px:
            touch.append(f)

    if len(touch) < 2:
        anchors = detected                      # không đủ mốc -> giữ nguyên (không bẻ thẳng)
    else:
        anchors = sorted(set(touch))
        if detected[0] < anchors[0]:
            anchors = [detected[0]] + anchors
        if detected[-1] > anchors[-1]:
            anchors = anchors + [detected[-1]]

    axs = np.array(anchors, float)
    ap = np.array([np.asarray(raw[i], float) for i in anchors])
    out = [None] * F
    for f in range(F):
        if f <= anchors[0]:
            out[f] = ap[0]
        elif f >= anchors[-1]:
            out[f] = ap[-1]
        else:
            out[f] = np.array([np.interp(f, axs, ap[:, 0]), np.interp(f, axs, ap[:, 1])])
    return out

In [ ]:
def compute_speeds(records, fps, max_player_ms=12.0, pos_smooth=5, spd_window=5, max_kmh=40.0):
    """Lượt 2b: tốc độ km/h theo tracker_id. Làm sạch id-swap + mượt vị trí -> bớt nhảy."""
    from collections import defaultdict
    tracks = defaultdict(list)
    for f, r in enumerate(records):
        for k in range(len(r['tid'])):
            if r['isref'][k]:
                continue
            p = r['pitch'][k]
            if not np.isnan(p).any():
                tracks[int(r['tid'][k])].append((f, np.asarray(p, float)))

    def moving_avg(xy, w):
        if w <= 1 or len(xy) < 3:
            return xy
        ker = np.ones(w) / w
        return np.stack([np.convolve(xy[:, 0], ker, 'same'),
                         np.convolve(xy[:, 1], ker, 'same')], 1)

    speed_by_frame = [dict() for _ in records]
    for tid, seq in tracks.items():
        seq.sort()
        frames = [seq[0][0]]; xy = [seq[0][1]]
        for f, p in seq[1:]:                         # bỏ cú nhảy phi lý (id-swap)
            df = f - frames[-1]
            if df > 0 and (np.linalg.norm(p - xy[-1]) / 100.0) / (df / fps) <= max_player_ms:
                frames.append(f); xy.append(p)
        frames = np.array(frames); xy = np.array(xy)
        if len(frames) < 2:
            continue
        xy_s = moving_avg(xy, pos_smooth)
        for j in range(len(frames)):                 # vi sai trung tâm trên cửa sổ
            j0 = max(0, j - spd_window); j1 = min(len(frames) - 1, j + spd_window)
            df = frames[j1] - frames[j0]
            if df > 0:
                d = np.linalg.norm(xy_s[j1] - xy_s[j0]) / 100.0
                speed_by_frame[frames[j]][tid] = min(d / (df / fps) * 3.6, max_kmh)
    return speed_by_frame

In [ ]:
def render_video(source_path, target_path, info, records, ball_pitch_s,
                 speed_by_frame, max_frames=None):
    """Lượt 2c: vẽ lại bằng dữ liệu đã hậu kỳ (KHÔNG infer lại)."""
    total = info.total_frames if max_frames is None else min(max_frames, info.total_frames)
    gen = sv.get_video_frames_generator(source_path)
    with sv.VideoSink(target_path, info) as sink:
        for f, frame in enumerate(tqdm(gen, total=total, desc='rendering')):
            if max_frames is not None and f >= max_frames:
                break
            r = records[f]
            annotated = frame.copy()

            if len(r['xyxy']):
                det = sv.Detections(xyxy=r['xyxy'], class_id=r['cid'].astype(int),
                                    tracker_id=r['tid'])
                annotated = ellipse_annotator.annotate(scene=annotated, detections=det)
                labels = []
                for k, tid in enumerate(r['tid']):
                    v = speed_by_frame[f].get(int(tid))
                    labels.append(f"{v:.0f} km/h" if (v is not None and not r['isref'][k])
                                  else f"#{int(tid)}")
                annotated = label_annotator.annotate(scene=annotated, detections=det, labels=labels)

            if r['ball_box'] is not None:
                bd = sv.Detections(xyxy=sv.pad_boxes(xyxy=r['ball_box'][None, :], px=10),
                                   class_id=np.array([BALL_ID]))
                annotated = triangle_annotator.annotate(scene=annotated, detections=bd)

            radar = voronoi = None
            pitch = r['pitch']
            if len(pitch):
                valid = ~np.isnan(pitch).any(axis=1)
                cid, isref = r['cid'], r['isref']
                t0 = pitch[valid & (cid == 0) & (~isref)]
                t1 = pitch[valid & (cid == 1) & (~isref)]
                rfp = pitch[valid & isref]
                bp = ball_pitch_s[f]
                bp_arr = bp[None, :] if bp is not None else np.empty((0, 2))
                if valid.any() or bp is not None:
                    radar = build_radar(bp_arr, t0, t1, rfp)
                    voronoi = build_voronoi(t0, t1, bp_arr)

            sink.write_frame(compose_frame(annotated, radar=radar, voronoi=voronoi))
    print('✅ Đã lưu video:', target_path)
    return target_path

In [ ]:
def run_video(source_path, target_path, max_frames=None, fit_stride=20,
              switch_ratio=0.5, touch_px=70.0):
    tc = fit_team_classifier(source_path, is_video=True, stride=fit_stride)
    info = sv.VideoInfo.from_video_path(source_path)
    resolver = TeamResolver(tc, window=int(round(info.fps)), switch_ratio=switch_ratio)

    print('— Lượt 1: phân tích —')
    _, records = analyze_video(source_path, resolver, max_frames=max_frames)
    print('— Lượt 2: hậu kỳ + vẽ —')
    ball_s = smooth_ball_path(records, info.fps, touch_px=touch_px)
    speeds = compute_speeds(records, info.fps)
    return render_video(source_path, target_path, info, records, ball_s, speeds,
                        max_frames=max_frames)

In [ ]:
SOURCE_VIDEO_PATH = "/content/2e57b9_0.mp4"
TARGET_VIDEO_PATH = "/content/result_full.mp4"

# Test nhanh ~150 frame trước; muốn render hết thì để max_frames=None
# touch_px: ngưỡng "chạm bóng" (px, ~1080p). Tăng nếu bóng còn vẽ vòng (sót lần chạm); giảm nếu bị bẻ thẳng nhầm.
run_video(SOURCE_VIDEO_PATH, TARGET_VIDEO_PATH, max_frames=150)

### Xem / tải video kết quả

In [ ]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open(TARGET_VIDEO_PATH, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=720 controls><source src="{data_url}" type="video/mp4"></video>')

In [ ]:
# Tải file về máy
from google.colab import files
files.download(TARGET_VIDEO_PATH)

## 11. Ghi chú & hướng tinh chỉnh

- **Bóng vẽ vòng cung (mục 10, lượt 2):** bóng trên map = đường gấp khúc nối các lần chạm bóng (nội suy thẳng giữa hai lần chạm). Còn sót vòng → **tăng `touch_px`** (bắt được lần chạm ở điểm sút/điểm rơi); bị bẻ thẳng nhầm khi rê dắt → **giảm `touch_px`**. Ngưỡng tính theo px ở 1080p, đổi độ phân giải thì chỉnh theo.
- **Tốc độ nhảy (mục 10):** đã làm mượt homography (EMA) + làm sạch id-swap + mượt vị trí. Còn nhiễu → tăng `pos_smooth`/`spd_window` trong `compute_speeds`, giảm `max_player_ms` để loại jump mạnh hơn, hoặc giảm `alpha` của `HomographySmoother`. Muốn label hiện cả id lẫn km/h → sửa dòng tạo `labels` trong `render_video`.
- **Chống flicker đội (mục 8):** còn nhấp nháy → tăng `switch_ratio` (0.6), giảm `fit_stride` (15).
- **Bộ nhớ:** lượt 1 giữ dữ liệu gọn từng frame trong RAM; video rất dài có thể tốn RAM — khi đó chia nhỏ theo đoạn rồi nối.
- **Bố cục:** `compose_frame` — `radar_scale`, `voronoi_scale`, `margin`, `alpha`, `voronoi_side`.
- **Ảnh (mục 9):** vẫn dùng `process_frame` (1 frame) nên không có tốc độ và không cần hậu kỳ bóng.
- **Lần sau dùng weights local:** thay 2 dòng ở mục *Load models* sang `YOLO(path)`.